In [1]:
from pathlib import Path
import os

from dotenv import load_dotenv
from PyPDF2 import PdfReader
import oracledb
import oci

from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_oracledb.vectorstores import oraclevs
from langchain_oracledb.vectorstores.oraclevs import OracleVS
from langchain_community.embeddings import OCIGenAIEmbeddings
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_core.documents import BaseDocumentTransformer, Document

print("Successfully imported libraries and modules")

# Load DB credentials from .env. DB_PASSWORD is also the PEM/wallet password.
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / ".env").exists() and Path("/Users/vavena/Documents/Playground/.env").exists():
    PROJECT_DIR = Path("/Users/vavena/Documents/Playground")

load_dotenv(PROJECT_DIR / ".env")

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
missing_env = [name for name, value in {"DB_USER": DB_USER, "DB_PASSWORD": DB_PASSWORD}.items() if not value]
if missing_env:
    raise RuntimeError(f"Missing required value(s) in .env: {', '.join(missing_env)}")

dsn = """(description= (retry_count=20)(retry_delay=3)(address=(protocol=tcps)(port=1522)(host=adb.us-phoenix-1.oraclecloud.com))(connect_data=(service_name=g10d1d163445a60_ragval_high.adb.oraclecloud.com))(security=(ssl_server_dn_match=yes)))"""

# Connect to the database
try:
    conn23c = oracledb.connect(user=DB_USER, password=DB_PASSWORD, dsn=dsn)
    print("Connection successful")
except Exception as e:
    print("Connection failed!")
    print(type(e).__name__, e)

# RAG Step 1 - Load the document and create pdf reader object 

pdf = PdfReader("./Oracle Cloud Infrastructure AI Foundations.pdf")

# RAG Step 2 - Transform the document to text

text = ""
for page in pdf.pages:
    text += page.extract_text()

print("You have transformed the PDF document to text format")



print("Number of pages:", len(pdf.pages))
print("PDF metadata:", pdf.metadata)

# RAG Step 3 - Chunk the text document into smaller chunks


from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = text_splitter.split_text(text)

#text_splitter = CharacterTextSplitter(separator=".", chunk_size=4000, chunk_overlap=100)
#chunks = text_splitter.split_text(text)


# Function to format and add metadata to Oracle 23ai Vector Store

def chunks_to_docs_wrapper(row: dict) -> Document:
    """
    Converts text into a Document object suitable for ingestion into Oracle Vector Store.
    - row (dict): A dictionary representing a row of data with keys for 'id', 'link', and 'text'.
    """
    metadata = {'id': row['id'], 'link': row['link']}
    return Document(page_content=row['text'], metadata=metadata)

# RAG Step 4 - Create metadata wrapper to store additional information in the vector store

"""
Converts a row from a DataFrame into a Document object suitable for ingestion into Oracle Vector Store.
- row (dict): A dictionary representing a row of data with keys for 'id', 'link', and 'text'.
"""

docs = [
    chunks_to_docs_wrapper({
        'id': str(page_num),
        'link': f'Page {page_num}',
        'text': text
    })
    for page_num, text in enumerate(chunks)
]


COMPARTMENT_OCID = "ocid1.compartment.oc1..aaaaaaaaat66um3xnwruivicqzcwelw3owwbce2ykrp72snonmlomngdb4ya"

endpoint = "https://inference.generativeai.us-phoenix-1.oci.oraclecloud.com"

config = oci.config.from_file("~/.oci/config", "DEFAULT")

signer = oci.auth.signers.SecurityTokenSigner(
    token=open(config["security_token_file"], "r").read(),
    private_key=oci.signer.load_private_key_from_file(config["key_file"]),
)

client = oci.generative_ai_inference.GenerativeAiInferenceClient(
    config={},
    signer=signer,
    service_endpoint=endpoint,
)

embed_model = OCIGenAIEmbeddings(
    model_id="openai.text-embedding-3-small",
    service_endpoint=endpoint,
    compartment_id=COMPARTMENT_OCID,
    auth_type="SECURITY_TOKEN",
    auth_profile="DEFAULT",
    auth_file_location="~/.oci/config",
)

test_vector = embed_model.embed_query("hello world")
print(type(test_vector))
print(len(test_vector))
print(test_vector[:5])


# RAG Step 5 - Using an embedding model, embed the chunks as vectors into Oracle Database 23ai.

#model_4db = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# RAG Step 6 - Configure the vector store with the model, table name, and using the indicated distance strategy for the similarity search and vectorize the chunks

knowledge_base = OracleVS.from_documents(
    docs,
    embed_model,
    client=conn23c,
    table_name="MY_DEMO",
    distance_strategy=DistanceStrategy.DOT_PRODUCT
)


with conn23c.cursor() as cursor:
    cursor.execute("""
        SELECT table_name
        FROM user_tables
        WHERE table_name = 'MY_DEMO'
    """)
    print(cursor.fetchall())

print("Docs expected:", len(docs))

with conn23c.cursor() as cursor:
    cursor.execute("SELECT COUNT(*) FROM MY_DEMO")
    print("Rows inserted:", cursor.fetchone()[0])    

with conn23c.cursor() as cursor:
    cursor.execute("""
        SELECT column_name, data_type
        FROM user_tab_columns
        WHERE table_name = 'MY_DEMO'
        ORDER BY column_id
    """)
    
    for row in cursor.fetchall():
        print(row)

results = knowledge_base.similarity_search(
    "What is Oracle AI Foundation?",
    k=3
)

print("Results:", len(results))

for i, doc in enumerate(results, start=1):
    print("\n--- Result", i, "---")
    print(doc.metadata)
    print(doc.page_content[:500])     

print("Chunks:", len(chunks))
print("Docs:", len(docs))

with conn23c.cursor() as cursor:
    cursor.execute("SELECT COUNT(*) FROM MY_DEMO")
    print("Rows in MY_DEMO:", cursor.fetchone()[0])

results = knowledge_base.similarity_search("AI Foundation", k=3)
print("Similarity results:", len(results))    

Successfully imported libraries and modules
Connection successful
You have transformed the PDF document to text format
Number of pages: 244
PDF metadata: {'/Author': 'Judith Meskill', '/CreationDate': "D:20220725090059-08'00'", '/ModDate': "D:20220725090059-08'00'", '/Creator': 'AH XSL Formatter V7.0 MR1 for Linux64 : 7.0.2.44154 (2020-04-06T11:51+09)', '/Producer': 'Antenna House PDF Output Library 7.0.1574', '/Title': 'User Guide ', '/Trapped': '/False'}


/Users/vavena/Documents/Playground/.venv-jupyter/lib/python3.14/site-packages/urllib3/poolmanager.py:329: FutureWarning: The 'strict' parameter is no longer needed on Python 3+. This will raise an error in urllib3 v3.0.
  warnings.warn(


<class 'list'>
1536
[-0.006793976, -0.03918457, 0.034118652, 0.0287323, -0.024810791]
[('MY_DEMO',)]
Docs expected: 186
Rows inserted: 186
('ID', 'RAW')
('TEXT', 'CLOB')
('METADATA', 'JSON')
('EMBEDDING', 'VECTOR')
Results: 3

--- Result 1 ---
{'id': '0', 'link': 'Page 0'}
Oracle® Retail AI Foundation Cloud
Services: AI Foundation
User Guide
Release 22.2.301.0
F58634–01
August 2022
Oracle Retail AI Foundation Cloud Services: AI Foundation User Guide, Release 22.2.301.0
F58634–01
Copyright © 2022, Oracle and/or its affiliates.
Primary Author: Judith Meskill
This software and related documentation are provided under a license agreement containing restrictions on
use and disclosure and are protected by intellectual property laws. Except as expressly permitted in your

--- Result 2 ---
{'id': '7', 'link': 'Page 7'}
•Oracle  Retail AI Foundation Cloud Services Administration  Guide
•Oracle Retail AI Foundation Cloud Services Implementation Guide
•Oracle Retail AI Foundation Cloud Services R